# 03 Weather data quality

Evidence for the decisions taken in notebook 02: that all four daily variables
share one grid, that the monthly files tile the window without gaps, and what
observing window each variable declares.

Downloads one month of each variable, about 290 MB, and deletes them at the end.
Each check prints a verdict.

In [1]:
import numpy as np
import pandas as pd

from fdh import weather as wx
from fdh import weather_checks as wc
from fdh.config import load_config, require

config = load_config()
BASE = require(config.weather.base_url, "weather.base_url")
VERSION = require(config.weather.version_directory, "weather.version_directory")
START = config.weather.window_start
END = config.weather.release_covers_to
print("token set:", wx.ensure_token())

findings = []          # one row per check, written to the quality report at the end
def record(check, verdict, detail):
    findings.append({"check": check, "verdict": verdict, "detail": detail})
    print(f"{verdict:6s}  {check}: {detail}")

token set: True


## 1. Which variables publish daily

The panel needs daily values.

In [2]:
daily = wx.daily_variables(BASE)
VARIABLES = sorted(name for name, has_day in daily.items() if has_day)
configured = require(config.weather.variables, "weather.variables")
not_daily = sorted(set(configured) - set(VARIABLES))

record("daily variables",
       "FAIL" if not_daily else "PASS",
       f"{len(VARIABLES)} of {len(daily)} publish daily: {VARIABLES}; "
       + (f"configured but not daily: {not_daily}" if not_daily
          else f"every configured variable {configured} publishes daily"))
pd.Series(daily, name="has_daily").to_frame()

PASS    daily variables: 4 of 15 publish daily: ['rainfall', 'tas', 'tasmax', 'tasmin']; every configured variable ['rainfall', 'tasmax', 'tasmin'] publishes daily


,has_daily
airfrost,False
groundfrost,False
hurs,False
psl,False
pv,False
raindays10mm,False
raindays1mm,False
rainfall,True
sfcWind,False
snowLying,False


## 2. One version directory per variable

In [3]:
versions = {name: wx.list_directories(f"{BASE}/{name}/day/") for name in VARIABLES}
as_configured = all(found == [f"{VERSION}/"] for found in versions.values())

record("version directories",
       "PASS" if as_configured else "CHECK",
       f"weather.version_directory {VERSION}; found {versions}")

PASS    version directories: weather.version_directory v20260512; found {'rainfall': ['v20260512/'], 'tas': ['v20260512/'], 'tasmax': ['v20260512/'], 'tasmin': ['v20260512/']}


## 3. The monthly files tile the window

Gaps mean missing weather days. Overlaps mean a day counted twice.

In [4]:
files = {name: wx.daily_files(BASE, name, VERSION, START, END) for name in VARIABLES}
coverage = pd.DataFrame(wc.contiguity_rows(files))
expected = wc.expected_months(START, END)
clean = (coverage.gaps.sum() == 0 and coverage.overlaps.sum() == 0
         and (coverage.distinct_months == coverage.files).all()
         and (coverage.distinct_months == expected).all())

record("file contiguity",
       "PASS" if clean else "CHECK",
       f"{expected} months expected per variable; gaps {coverage.gaps.sum()}, "
       f"overlaps {coverage.overlaps.sum()}, duplicate months "
       f"{(coverage.files - coverage.distinct_months).sum()}, "
       f"months per variable {sorted(set(coverage.distinct_months))}")
coverage

PASS    file contiguity: 189 months expected per variable; gaps 0, overlaps 0, duplicate months 0, months per variable [189]


,variable,files,distinct_months,first,last,gaps,overlaps,gap_spans,overlap_spans,gigabytes
0,rainfall,189,189,2010-04-01,2025-12-31,0,0,[],[],13.6
1,tas,189,189,2010-04-01,2025-12-31,0,0,[],[],13.7
2,tasmax,189,189,2010-04-01,2025-12-31,0,0,[],[],13.7
3,tasmin,189,189,2010-04-01,2025-12-31,0,0,[],[],13.8


## 4. Every variable offers the same months

An unbalanced panel would silently shorten whichever covariate is missing.

In [5]:
balance = pd.DataFrame(wc.coverage_rows(files))
short = balance.loc[balance.missing.map(len) > 0]

record("month coverage",
       "PASS" if short.empty else "CHECK",
       f"{balance.months.max()} months offered across {len(VARIABLES)} variables"
       + ("; every variable offers all of them" if short.empty
          else f"; missing {dict(zip(short.variable, short.missing, strict=True))}"))
balance

PASS    month coverage: 189 months offered across 4 variables; every variable offers all of them


,variable,months,missing
0,rainfall,189,[]
1,tas,189,[]
2,tasmax,189,[]
3,tasmin,189,[]


## 5. Pull one month of each variable

April 2010 is the first month of the weather window. The files are deleted in the
last cell.

In [6]:
probe_dir = config.paths.resolved("raw_dir") / "weather" / "probe"
probes = {}
for name, items in files.items():
    probes[name] = wx.download(items[0], probe_dir)
    print(f"{name}: {probes[name].name} {probes[name].stat().st_size / 1e6:.0f} MB")

described = {name: wc.describe_file(path) for name, path in probes.items()}

rainfall: rainfall_hadukgrid_uk_1km_day_20100401-20100430.nc 63 MB
tas: tas_hadukgrid_uk_1km_day_20100401-20100430.nc 72 MB
tasmax: tasmax_hadukgrid_uk_1km_day_20100401-20100430.nc 72 MB
tasmin: tasmin_hadukgrid_uk_1km_day_20100401-20100430.nc 73 MB


## 6. The declared observing window

The window decides how a weather day aligns with an incident day. Each file
states it twice: in `time_bnds`, and in the text of `cell_methods` and `comment`.
A file whose text says nothing about a 09:00 day is reported as not stated, not
as a disagreement.

In [7]:
windows = pd.DataFrame(wc.window_rows(list(described.values())))
disagree = windows.loc[windows.agreement == "disagree", "variable"].tolist()
unstated = windows.loc[windows.agreement == "not stated", "variable"].tolist()
one_window = windows.bounds_start_hours.astype(str).nunique() == 1

record("observing window",
       "PASS" if one_window and not disagree else "CHECK",
       ("one window across variables" if one_window else "variables declare different windows")
       + (f"; bounds and text disagree for {disagree}" if disagree else "")
       + (f"; text states no window for {unstated}" if unstated else ""))
windows

CHECK   observing window: variables declare different windows; bounds and text disagree for ['tasmax', 'tasmin']; text states no window for ['rainfall', 'tas']


,variable,cell_methods,comment,bounds_start_hours,bounds_span_hours,text_states_0900,bounds_start_0900,agreement
0,rainfall,time: sum,None,[9],[24.0],False,True,not stated
1,tas,time: mid_range,None,[0],[24.0],False,False,not stated
2,tasmax,time: maximum (comment: observing period is 09...,None,[0],[24.0],True,False,disagree
3,tasmin,time: minimum (comment: observing period is 09...,None,[0],[24.0],True,False,disagree


## 7. One grid for all four variables

The cell-to-service mask is built once and applied to every file.

In [8]:
grids = pd.DataFrame(wc.grid_rows(list(described.values())))
reference = described[VARIABLES[0]]
aligned = grids.x_matches.all() and grids.y_matches.all()

record("grid alignment",
       "PASS" if aligned else "FAIL",
       f"shape {reference['shape'][1:]}, {reference['valid_cells']:,} valid cells "
       f"({reference['valid_share']:.1%}); masks identical: {grids.mask_matches.all()}")
grids

PASS    grid alignment: shape (1450, 900), 245,077 valid cells (18.8%); masks identical: True


,variable,shape,x_matches,y_matches,mask_matches,valid_cells,valid_share,mask_varies_within_month
0,rainfall,"(1450, 900)",True,True,True,245077,0.1878,False
1,tas,"(1450, 900)",True,True,True,245077,0.1878,False
2,tasmax,"(1450, 900)",True,True,True,245077,0.1878,False
3,tasmin,"(1450, 900)",True,True,True,245077,0.1878,False


## 8. Metadata contract and value ranges

Provenance strings go into the methodology. The ranges are a sanity check on
units.

In [9]:
values = pd.DataFrame(wc.range_rows(list(described.values())))
provenance_consistent = values.source.nunique() == 1 and values.version.nunique() == 1
within = values.within_expected.eq(True).all()

record("metadata contract",
       "PASS" if provenance_consistent and within else "CHECK",
       f"source {values.source.iloc[0]}, version {values.version.iloc[0]}; "
       f"values within expected range: {within}")
values

PASS    metadata contract: source HadUK-Grid_v1.3.2.0, version v20260512; values within expected range: True


,variable,units,fill,source,version,calendar,steps,min,max,within_expected,expected
0,rainfall,mm,1.000000e+20,HadUK-Grid_v1.3.2.0,v20260512,standard,30,0.00,100.07,True,0.0 to 500.0
1,tas,degC,1.000000e+20,HadUK-Grid_v1.3.2.0,v20260512,standard,30,-6.58,17.81,True,-30.0 to 45.0
2,tasmax,degC,1.000000e+20,HadUK-Grid_v1.3.2.0,v20260512,standard,30,-6.24,23.07,True,-25.0 to 45.0
3,tasmin,degC,1.000000e+20,HadUK-Grid_v1.3.2.0,v20260512,standard,30,-9.59,16.61,True,-35.0 to 35.0


## 9. Cross-variable consistency

`tasmax` for day D covers 09:00 D to 09:00 D+1 and `tasmin` for day D covers
09:00 D-1 to 09:00 D, as their `cell_methods` state, so the minimum inside the
window of `tasmax[D]` is `tasmin[D+1]`. That is the pair compared; the same-day
count is shown for reference. The check stops if either file states another
period.

Failures left at the matched alignment are attributed provisionally to the two
grids being interpolated separately, which is unverified against Hollis et al.
(2019). The verdict compares the failing share with `weather.ordering_tolerance`.

In [10]:
from netCDF4 import Dataset

stack = {}
for name, path in probes.items():
    with Dataset(path) as ds:
        var = next(v for v in ds.variables.values() if v.ndim == 3)
        stack[name] = np.ma.masked_invalid(var[:])

ordering = wc.temperature_ordering(
    stack["tasmin"],
    stack["tasmax"],
    described["tasmin"]["cell_methods"],
    described["tasmax"]["cell_methods"],
)
tolerance = config.weather.ordering_tolerance
if tolerance is None:
    verdict, basis = "CHECK", "weather.ordering_tolerance is undecided"
else:
    verdict = "PASS" if ordering["failing_share"] <= tolerance else "CHECK"
    basis = f"tolerance {tolerance:.4%}"

record("temperature ordering",
       verdict,
       f"tasmin[D+1] > tasmax[D] in {ordering['failing']:,} of "
       f"{ordering['cell_days']:,} cell-days ({ordering['failing_share']:.4%}); "
       f"median excess {ordering['median_excess']:.2f} C, "
       f"max {ordering['max_excess']:.2f} C; same-day comparison "
       f"{ordering['same_day_failing']:,} of {ordering['same_day_cell_days']:,}; {basis}")
ordering

CHECK   temperature ordering: tasmin[D+1] > tasmax[D] in 1,273 of 7,107,233 cell-days (0.0179%); median excess 0.53 C, max 9.92 C; same-day comparison 1,985 of 7,352,310; weather.ordering_tolerance is undecided


{'cell_days': 7107233,
 'failing': 1273,
 'failing_share': 0.00017911330612068016,
 'median_excess': 0.5292124718198963,
 'max_excess': 9.918223501019899,
 'same_day_cell_days': 7352310,
 'same_day_failing': 1985}

## 10. tas is a definition, not an independent field

cell_methods on tas is "time: mid_range", so the published mean is the midpoint
of tasmax and tasmin by construction. Comparing the two confirms that definition
holds in the data. Exact agreement is the expected result, and a difference would
mean the files do not match their metadata.

In [11]:
valid = ~stack["tasmax"].mask & ~stack["tasmin"].mask & ~stack["tas"].mask
derived = (stack["tasmax"] + stack["tasmin"]) / 2
difference = (derived - stack["tas"])[valid]
exact = bool((np.abs(difference) < 1e-9).all())
record("tas is mid_range",
       "PASS" if exact else "FAIL",
       f"published tas equals (tasmax + tasmin) / 2 exactly: {exact}; "
       f"max absolute difference {float(np.abs(difference).max()):.2e} over "
       f"{valid.sum():,} cell-days. tas carries no information beyond the pair "
       f"and is dropped from the download.")

PASS    tas is mid_range: published tas equals (tasmax + tasmin) / 2 exactly: True; max absolute difference 0.00e+00 over 7,352,310 cell-days. tas carries no information beyond the pair and is dropped from the download.


## Verdicts

In [12]:
report = pd.DataFrame(findings)
report

,check,verdict,detail
0,daily variables,PASS,"4 of 15 publish daily: ['rainfall', 'tas', 'ta..."
1,version directories,PASS,weather.version_directory v20260512; found {'r...
2,file contiguity,PASS,"189 months expected per variable; gaps 0, over..."
3,month coverage,PASS,189 months offered across 4 variables; every v...
4,observing window,CHECK,variables declare different windows; bounds an...
5,grid alignment,PASS,"shape (1450, 900), 245,077 valid cells (18.8%)..."
6,metadata contract,PASS,"source HadUK-Grid_v1.3.2.0, version v20260512;..."
7,temperature ordering,CHECK,"tasmin[D+1] > tasmax[D] in 1,273 of 7,107,233 ..."
8,tas is mid_range,PASS,published tas equals (tasmax + tasmin) / 2 exa...


In [13]:
out = config.paths.resolved("reports_dir")
out.mkdir(parents=True, exist_ok=True)
report.to_csv(out / "weather_quality_checks.csv", index=False)
print(f"wrote {out / 'weather_quality_checks.csv'}")

wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\reports\weather_quality_checks.csv


## Clean up

The probe files are 290 MB and serve no further purpose.

In [14]:
for path in probes.values():
    path.unlink(missing_ok=True)
print("probe files removed")

probe files removed
